# Vectorizing the ray-cast functions — behaviour-equivalence + speed

`compute_visibility`, `compute_achievable_cells`, and `update_covered_mask` all ray-walk the
grid with pure-Python triple loops (`num_rays × max_range × candidates`). This is the hottest
path in the planner (`compute_all_visibility` ≈ 99 % of `plan_waypoints` time). This notebook:

1. **golden-records** the current (loop) output of each function on real + synthetic maps and
   the 9 sensor configs from `TestCameraPlanningScanConsistency`;
2. implements **vectorized** replacements;
3. asserts **exact** equivalence (identical sets / identical covered_mask / identical ratio) —
   the bar the source change must clear before it is made;
4. benchmarks the speedup.

> Nothing here modifies the source. It is the safety net: the vectorized versions must reproduce
> the loop output **bit-for-bit** (same discrete-angle rounding, same per-ray stop) or the change
> is not made. The tightest constraint is `TestCameraPlanningScanConsistency`, which requires
> every cell `compute_visibility` sees at angle k·step to also be reached by `update_covered_mask`
> at the same angles — so the vectorized code must keep the identical `floor(base+r·trig+0.5)`
> rounding and the `leniency = step/2` window.

In [ ]:
import sys, math, time
from pathlib import Path
import numpy as np
import yaml
from PIL import Image

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and not (ROOT / "docker-compose.yml").is_file():
    ROOT = ROOT.parent
ASSETS = ROOT / "assets"
sys.path.insert(0, str(ROOT / "ros2_ws/src/navigation/exploration"))
import exploration.explore_costmap_map as ecm
from exploration.explore_costmap_map import (
    build_map_data, generate_candidates,
    compute_visibility, compute_achievable_cells, update_covered_mask,
)
print("repo root:", ROOT)

## Test maps + sensor configs

Real maps + the ring/L synthetics (same as the other studies), and the 9 `(num_rays, fov,
range, increment)` configs from `TestCameraPlanningScanConsistency`.

In [ ]:
def load_real(name):
    d = ASSETS / name
    yml = yaml.safe_load(open(d / "map.yaml"))
    arr = np.array(Image.open(d / "map.pgm"), dtype=np.uint8)
    res = float(yml["resolution"])
    return build_map_data(1.0 - arr / 255.0, res, float(yml["origin"][0]),
                          float(yml["origin"][1]), inflation_radius_m=0.9)

def _md_from_free(free, res=0.05, infl_px=1.0):
    return build_map_data((~free).astype(float), res, 0.0, 0.0, inflation_radius_m=infl_px * res)

def make_ring():
    free = np.zeros((160, 160), bool); free[20:140, 20:140] = True; free[55:105, 55:105] = False
    return _md_from_free(free)

def make_L():
    free = np.zeros((160, 160), bool); free[20:60, 20:140] = True; free[20:140, 20:60] = True
    return _md_from_free(free)

MAPS = {"lab_ghent": load_real("lab_ghent"), "lab_05": load_real("lab_05"),
        "ring": make_ring(), "L": make_L()}

# (num_rays, fov_deg, max_range_m, increment_deg) — from TestCameraPlanningScanConsistency
CONFIGS = [(180,87,6.0,30),(360,87,6.0,30),(720,87,6.0,30),(360,36,6.0,30),
           (360,140,6.0,30),(360,87,3.0,30),(360,87,9.0,30),(360,87,6.0,10),(360,87,6.0,45)]
print(f"{len(MAPS)} maps, {len(CONFIGS)} sensor configs")

## Vectorized implementations

Key idea: for a set of ray angles, build the `(n_angles × max_range)` arrays of `col/row` with
the **identical** `floor(base + r·trig + 0.5)` rounding, look up the masks by fancy indexing, then
find the first blocker per ray with a cumulative-OR so everything at/after the block is dropped.
The per-cell semantics (frontier record / covered predicate / free-mask write) are applied on the
surviving cells, matching each loop's rules exactly.

In [ ]:
# IMPORTANT — trig must use math.cos/sin per angle, NOT np.cos/np.sin.
# numpy and the C math lib disagree by ~1 ULP on some angles; at a cell boundary that 1e-16
# flips floor(base + r·trig + 0.5) by one pixel. Since the loops use math.*, the vectorized
# versions must too, or ~1 cell per candidate drifts (which breaks exact-set tests). We keep the
# per-angle trig scalar (cheap, num_rays values) and vectorize only the r-stepping + mask lookups.

def _dirs(num_rays):
    """Per-angle (cos, sin) via math.* — bit-identical to the loops."""
    cos_a = np.array([math.cos(2.0 * math.pi * i / num_rays) for i in range(num_rays)])
    sin_a = np.array([math.sin(2.0 * math.pi * i / num_rays) for i in range(num_rays)])
    return cos_a, sin_a


def _ray_grid(col0, row0, cos_a, sin_a, max_range_px):
    """(n_angles, max_range) integer col/row arrays, identical rounding to the loops."""
    r = np.arange(1, max_range_px + 1)                                # (R,)
    col = np.floor(col0 + np.outer(cos_a, r) + 0.5).astype(np.intp)   # (A,R)
    row = np.floor(row0 - np.outer(sin_a, r) + 0.5).astype(np.intp)   # (A,R)
    return col, row


def _first_block_mask(col, row, occupied, unknown, H, W):
    """Boolean (A,R): 'alive' = cell strictly before the first stop on its ray.
    Stop = out-of-bounds OR occupied OR unknown. Also returns the frontier flag (the unknown
    cell that stopped the ray) and clipped rr/cc for safe mask lookups."""
    inb = (col >= 0) & (col < W) & (row >= 0) & (row < H)
    cc = np.clip(col, 0, W - 1); rr = np.clip(row, 0, H - 1)
    occ = inb & occupied[rr, cc]
    unk = inb & unknown[rr, cc]
    stop = (~inb) | occ | unk                        # this cell terminates the ray
    stops_before = np.cumsum(stop, axis=1) - stop    # #stops strictly before this index
    alive = (stops_before == 0) & (~stop)            # free, reachable, ray not yet blocked
    first_stop = (np.cumsum(stop, axis=1) == 1) & stop
    frontier_here = first_stop & unk                 # the unknown cell that stopped the ray
    return alive, frontier_here, rr, cc


def compute_visibility_vec(candidate, md, max_range_px, num_rays=360):
    col0, row0 = candidate
    H, W = md.occupied_mask.shape
    cos_a, sin_a = _dirs(num_rays)
    col, row = _ray_grid(col0, row0, cos_a, sin_a, max_range_px)
    alive, frontier_here, rr, cc = _first_block_mask(col, row, md.occupied_mask, md.unknown_mask, H, W)
    cov_sel = alive & (~md.covered_mask[rr, cc])          # coverage = alive AND not covered
    coverage = set(zip(col[cov_sel].tolist(), row[cov_sel].tolist()))
    frontiers = set(zip(col[frontier_here].tolist(), row[frontier_here].tolist()))
    return coverage, frontiers


def compute_achievable_cells_vec(candidates, md, max_range_m, num_rays=360):
    max_range_px = max(1, int(max_range_m / md.resolution))
    H, W = md.occupied_mask.shape
    cos_a, sin_a = _dirs(num_rays)
    achievable = set()
    for col0, row0 in candidates:
        col, row = _ray_grid(col0, row0, cos_a, sin_a, max_range_px)
        alive, _, _, _ = _first_block_mask(col, row, md.occupied_mask, md.unknown_mask, H, W)
        achievable.update(zip(col[alive].tolist(), row[alive].tolist()))
    return achievable


def update_covered_mask_vec(md, robot_col, robot_row, heading_deg, fov_deg, max_range_px, num_rays=360):
    H, W = md.occupied_mask.shape
    step_deg = 360.0 / num_rays
    leniency_deg = step_deg / 2.0
    k_start = math.ceil((heading_deg - fov_deg / 2.0 - leniency_deg) / step_deg)
    k_end = math.floor((heading_deg + fov_deg / 2.0 + leniency_deg) / step_deg)
    ks = range(k_start, k_end + 1)
    # math trig per k (matches the loop's `math.radians(k*step_deg)` exactly)
    cos_a = np.array([math.cos(math.radians(k * step_deg)) for k in ks])
    sin_a = np.array([math.sin(math.radians(k * step_deg)) for k in ks])
    col, row = _ray_grid(robot_col, robot_row, cos_a, sin_a, max_range_px)
    alive, _, rr, cc = _first_block_mask(col, row, md.occupied_mask, md.unknown_mask, H, W)
    write = alive & md.free_mask[rr, cc]
    md.covered_mask[row[write], col[write]] = True
    total_free = int(np.sum(md.free_mask))
    return 1.0 if total_free == 0 else float(np.sum(md.covered_mask & md.free_mask)) / total_free

## Equivalence check — vectorized MUST equal the loops, exactly

For each map × config × candidate we compare loop vs vectorized outputs as exact sets / arrays.
Any mismatch is a blocker: the source change is not made until this reports **0 mismatches**.

In [ ]:
mismatches = {"visibility": 0, "achievable": 0, "covered_mask": 0, "ratio": 0}
checked = 0

for mname, md in MAPS.items():
    for (num_rays, fov, rng_m, inc) in CONFIGS:
        max_range_px = max(1, int(rng_m / md.resolution))
        step_px = max(1, int(rng_m / md.resolution))
        cands = generate_candidates(md.navigable_mask, step_px)
        # cap candidates per (map,config) so the exhaustive sweep stays tractable
        cands = cands[:8]

        # --- achievable (whole candidate set) ---
        md.covered_mask[:] = False
        a_loop = compute_achievable_cells(cands, md, rng_m, num_rays)
        a_vec = compute_achievable_cells_vec(cands, md, rng_m, num_rays)
        if a_loop != a_vec:
            mismatches["achievable"] += 1

        for cand in cands:
            col0, row0 = cand
            # --- visibility (with a non-trivial covered_mask to exercise the 'not covered' path) ---
            md.covered_mask[:] = False
            md.covered_mask[row0, :col0] = True   # arbitrary pre-covered stripe
            cov_l, fr_l = compute_visibility(cand, md, max_range_px, num_rays)
            cov_v, fr_v = compute_visibility_vec(cand, md, max_range_px, num_rays)
            if cov_l != cov_v or fr_l != fr_v:
                mismatches["visibility"] += 1

            # --- update_covered_mask (compare resulting mask + returned ratio) ---
            for h in (0.0, 37.0, 123.5, 270.0):
                md.covered_mask[:] = False
                r_loop = update_covered_mask(md, col0, row0, h, fov, max_range_px, num_rays)
                m_loop = md.covered_mask.copy()
                md.covered_mask[:] = False
                r_vec = update_covered_mask_vec(md, col0, row0, h, fov, max_range_px, num_rays)
                m_vec = md.covered_mask.copy()
                if not np.array_equal(m_loop, m_vec):
                    mismatches["covered_mask"] += 1
                if abs(r_loop - r_vec) > 1e-12:
                    mismatches["ratio"] += 1
            checked += 1

print(f"candidates checked: {checked}")
print("mismatches:", mismatches)
assert all(v == 0 for v in mismatches.values()), "NOT equivalent — do NOT change the source"
print("\nALL EQUIVALENT ✓  — vectorized output is bit-for-bit identical to the loops.")

## Speedup

In [ ]:
import timeit
md = MAPS["lab_ghent"]
cand = generate_candidates(md.navigable_mask, max(1, int(6.0 / md.resolution)))[0]
mrp = max(1, int(6.0 / md.resolution))
md.covered_mask[:] = False

t_loop = min(timeit.repeat(lambda: compute_visibility(cand, md, mrp, 360), repeat=3, number=10)) / 10
t_vec  = min(timeit.repeat(lambda: compute_visibility_vec(cand, md, mrp, 360), repeat=3, number=10)) / 10
print(f"compute_visibility (lab_ghent, 1 candidate, num_rays=360):")
print(f"  loop:       {t_loop*1e3:8.3f} ms")
print(f"  vectorized: {t_vec*1e3:8.3f} ms   ({t_loop/t_vec:.1f}× faster)")

## Outcome

Fill in after running:
- If the equivalence cell prints **ALL EQUIVALENT ✓** across all maps × 9 configs, the vectorized
  implementations are a safe drop-in for the three source functions (they reproduce the loops
  bit-for-bit, including the discrete-angle rounding the `TestCameraPlanningScanConsistency`
  pipeline depends on).
- The speedup cell quantifies the win on the hottest function.

Only after both are satisfactory should the source be changed — then re-run the real suite
(`TestComputeVisibility`, `TestComputeAchievableCells`, `TestUpdateCoveredMask`,
`TestCameraPlanningScanConsistency`) as the final gate. Recorded as `docs/TODO_exploration_refactor.md` §2c.